# Setting Up

In [1]:
import torch
import numpy as np
import pandas as pd
print("CUDA available:", torch.cuda.is_available())

CUDA available: True


In [2]:
!nvidia-smi

Fri Jul 24 11:01:46 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 610.74                 KMD Version: 610.74        CUDA UMD Version: 13.3     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                  Driver-Model | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA GeForce RTX 4080 ...  WDDM  |   00000000:01:00.0 Off |                  N/A |
| N/A   55C    P0             27W /  160W |       0MiB /  12282MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

# Manipulating Tensors

## 1. Basic Operations

### Tensor with Scaler

#### Addition

In [3]:
T = torch.tensor([1,2,3])
T + 10

tensor([11, 12, 13])

In [4]:
torch.add(T, 10) #not in-place

tensor([11, 12, 13])

#### Multiplication

In [5]:
T*10

tensor([10, 20, 30])

In [6]:
torch.multiply(T, 10) #not in-place

tensor([10, 20, 30])

#### Division

In [7]:
T/10

tensor([0.1000, 0.2000, 0.3000])

In [8]:
torch.divide(T, 10) #not in-place

tensor([0.1000, 0.2000, 0.3000])

#### Subtraction

In [9]:
T-10

tensor([-9, -8, -7])

In [10]:
torch.subtract(T, 10) #not in-place

tensor([-9, -8, -7])

### Tensor with Tensor

#### Addition

##### To add tensors together, they have to be either:

- ##### Same shape (all dimensions are the same)

In [11]:
a = torch.tensor([[1,2,3],[4,5,6]])
b = torch.tensor([[4,5,6],[7,8,9]])
a + b

tensor([[ 5,  7,  9],
        [11, 13, 15]])

- ##### When they are not equal in a certain dimension, one of the dimensions is equal to one (broadcasting)

In [12]:
a = torch.tensor([[[1,2,3],[10,11,12]]])
print(a.shape)

b= torch.tensor([[[4,5,6]],[[7,8,9]]])
print(b.shape)

a+b

torch.Size([1, 2, 3])
torch.Size([2, 1, 3])


tensor([[[ 5,  7,  9],
         [14, 16, 18]],

        [[ 8, 10, 12],
         [17, 19, 21]]])

- ##### One tesnor has fewer dimensions (this is what makes the tensor+scaler work)

In [13]:
a = torch.tensor([[1,2,3],[10,11,12]])
print(a.shape)
b = torch.tensor([4,5,6])
print(b.shape)

a+b

torch.Size([2, 3])
torch.Size([3])


tensor([[ 5,  7,  9],
        [14, 16, 18]])

#### Multiplication

##### Element Wise Multiplication

[1 * 1, 2 * 2, 3 * 3] = [1, 4, 9]

can be only done if two tensors have the same dimension or broadcastable

In [14]:
a = torch.tensor([[1,2,3],[10,11,12]])
print(a.shape)
b= torch.tensor([[4,5,6],[7,8,9]])
print(b.shape)

a*b

torch.Size([2, 3])
torch.Size([2, 3])


tensor([[  4,  10,  18],
        [ 70,  88, 108]])

##### Matrix Multiplication (More Commonly Used)


When you use torch.matmul(A, B) or A @ B with tensors that have more than 2 dimensions, PyTorch splits the rules in half:

- The Matrix Rule (The Last 2 Dimensions): PyTorch looks strictly at the last two dimensions of both tensors. These must follow the strict linear algebra rule: inner dimensions must match.

- The Broadcast Rule (Everything Else): PyTorch looks at all the dimensions in front of the last two and treats them as "Batch" dimensions. It applies the standard right-to-left broadcasting rules to these batch dimensions!

Real-World Example:
Imagine you have a batch of 32 samples (flattened to 64 features each), and a single weight matrix.

- Batch A: [32, 10, 64] (32 batches, 10 sequences, 64 features)

- Weights B: [64, 128] (64 input features, 128 output features)

How torch.matmul(A, B) handles this:

- The Matrices (Last 2 Dims): [..., 10, 64] @ [..., 64, 128]. Inner dimensions match (64 == 64). The result for the matrices is [10, 128].

- The Batches (Front Dims): A has batch [32]. B has no batch dimensions, so PyTorch treats it as [1]. 32 vs 1 broadcasts perfectly to 32.

- Final Result Shape: [32, 10, 128]

In [15]:
%%time
a= torch.randn(200,300)
b= torch.randn(300,200)
torch.matmul(a,b)

CPU times: total: 0 ns
Wall time: 13.1 ms


tensor([[  9.8781,  18.4551,  13.4659,  ...,  -8.8743,  23.1372,   4.3679],
        [-17.0574, -18.0414,   6.0438,  ..., -15.1411,  17.5402,  10.0223],
        [ -4.4556,   8.8180,   5.5772,  ...,  -3.3640,  -6.5339,  33.1892],
        ...,
        [  3.2285, -16.1307, -19.1617,  ...,  24.8252,  27.5544,  28.2400],
        [-21.7366,  -9.6407,   8.8987,  ..., -17.5672, -25.0822, -24.3141],
        [  2.0687,  19.2451,   6.5516,  ..., -29.5492,  18.2653,   7.5087]])

In [16]:
%%time
a= torch.randn(200,300,device='cuda')
b= torch.randn(300,200,device='cuda')
a@b

CPU times: total: 4.39 s
Wall time: 651 ms


tensor([[ 11.2448, -14.7634, -13.8371,  ..., -29.4334,  15.0742,  30.1647],
        [-35.6849, -14.6980,   1.6447,  ..., -24.5562,  14.3937,  28.9549],
        [-18.3042,  37.2284,  -7.4725,  ...,  29.4219,   0.5209, -24.0746],
        ...,
        [ 11.3850, -13.3636, -33.6611,  ...,  13.5947,  15.9395, -12.0648],
        [  2.2346,   1.3901,  34.1851,  ...,   2.0784,  -9.5344, -11.8292],
        [  1.9212, -13.3145, -11.5806,  ..., -25.3478,  -3.6299, -19.5221]],
       device='cuda:0')

### Some Best Practicces

In [17]:
x = torch.ones(2, 2)

# Standard (Allocates new memory)
y = x.add(5)  

# In-Place (Modifies 'x' directly, saves memory)
x.add_(5)


tensor([[6., 6.],
        [6., 6.]])

##### Never use in-place operations on tensors that require gradients (requires_grad=True) during your forward pass. PyTorch's autograd engine needs the original tensor values to calculate derivatives. If you overwrite them in-place, .backward() will immediately crash.

Standard float32 matrix multiplication is accurate but slightly slow. Nvidia created a format called TF32 that uses 19 bits instead of 32 bits for the math, delivering almost the exact same accuracy but running 2x to 3x faster.

The Best Practice: Add this to the top of your main training script to get free speed on matrix multiplications:

In [18]:
torch.backends.cuda.matmul.allow_tf32 = True

## 2. Shape-Shifting

### Squeeze & Unsqueeze

They never change the physical memory in any way

In [54]:
state = torch.randn(64)

# Insert a batch dimension at index 0
batch_state = state.unsqueeze(dim=0) # dim is required to specify where to insert the new dimension

batch_state.shape

torch.Size([1, 64])

In [55]:
x = torch.randn(1, 64, 1)

# Global squeeze: removes all size-1 dims
y = x.squeeze()
print(y.shape)  # torch.Size([64])

# Targeted squeeze: removes dim 0 only
z = x.squeeze(dim=0)
print(z.shape)  # torch.Size([64, 1])

torch.Size([64])
torch.Size([64, 1])


#### Some Best Practices

never call .squeeze() without specifying a dimension in research code, as it can to unpredictive behavior

### Reshape & View

#### Reshape actually changes the shape of the tensor

In [19]:
x = torch.arange(1., 8.)
x, x.shape

(tensor([1., 2., 3., 4., 5., 6., 7.]), torch.Size([7]))

In [20]:
x = torch.arange(1., 8.)
x, x.shape

(tensor([1., 2., 3., 4., 5., 6., 7.]), torch.Size([7]))

In [21]:
X = torch.randn(20,50)
X_reshaped = X.reshape(10,100)
X_reshaped.shape


torch.Size([10, 100])

#### Flatten is a way to completely flatten a tensor or a dim in tensor

In [22]:
X.flatten().shape

torch.Size([1000])

#### Most of the times you will want to squash or flatten certain dimensions while leaving others

In [23]:
CNN_output=torch.randn(20, 50, 28, 28)  # Example CNN output with shape (batch_size, channels, height, width)
# Flatten the CNN output to (batch_size, channels * height * width)

CNN_reshaped = CNN_output.reshape(CNN_output.shape[0], -1)
# or 
CNN_flattened = CNN_output.flatten(start_dim=1)

CNN_reshaped.shape, CNN_flattened.shape

(torch.Size([20, 39200]), torch.Size([20, 39200]))

#### Reshape tries to run View at the beginning, if it fails it creates a new copy of the data
#### View doesnt play around with the physical data, making it extremely fast but only works with contigous data

In [24]:
X = torch.randn(20,50)
X.view(10,100)

tensor([[-4.6961e-01,  9.1912e-01, -1.3012e-01, -1.5905e-02,  6.6054e-02,
          2.0246e-01, -2.4755e+00, -1.1601e+00,  2.0918e+00, -9.6022e-01,
          7.8284e-01,  8.6021e-01,  2.8388e+00, -1.4161e-01, -5.3238e-01,
          5.9904e-01, -1.7482e-01, -1.7247e-01, -6.8277e-01,  2.2837e+00,
         -1.3405e+00, -3.2189e-01,  6.7779e-02, -2.0516e-01,  4.4871e-01,
         -8.9268e-01, -2.2251e-01,  1.7236e+00,  1.1727e+00, -3.0024e-02,
         -8.7099e-01,  1.5389e+00, -1.4250e+00,  2.1492e+00,  1.3947e+00,
          6.8848e-02,  3.8796e-01,  7.5672e-01,  7.0099e-01,  1.1562e-01,
         -4.8448e-01,  9.9046e-01, -2.2561e-01, -1.4674e-01, -1.7050e+00,
         -1.9051e-01, -6.0739e-01, -3.7777e-01, -4.2885e-01, -6.2239e-01,
          5.2233e-01,  1.8456e+00, -8.7983e-03,  2.3449e+00, -1.6598e-01,
         -3.1659e-01,  9.1510e-01, -2.1283e-01, -1.4971e+00, -4.8527e-01,
         -1.6798e+00, -8.6900e-01,  5.4494e-02,  1.5394e+00,  1.2103e+00,
          3.4879e-01, -2.1362e-01, -6.

#### Some Best Practices

Avoid Hardcoding batch dimension, rather use -1

In [25]:
x = torch.randn(32, 64, 8, 8) # Normal batch
last_batch = torch.randn(8, 64, 8, 8) # The leftover batch

# (Crashes on the last batch)
# bad = last_batch.reshape(32, 4096) # RuntimeError!

# (Safe forever)
last_batch.reshape(-1, 4096).shape # Automatically infers 8 batches!

torch.Size([8, 4096])

### Transpose & Permute

#### Transpose doesnt change the physical memory, but it changes the reading map, that is why it is fast

In [26]:
X = torch.randn(200, 300)
X.T.shape

torch.Size([300, 200])

##### On Matrices with dims>2 transpose is not recommended rather permute

In [38]:
X = torch.randn(1,2,3,4,5)
X.T.shape

C:\Users\Karim\AppData\Local\Temp\ipykernel_17300\377804360.py:2: UserWarning: The use of `x.T` on tensors of dimension other than 2 to reverse their shape is deprecated and it will throw an error in a future release. Consider `x.mT` to transpose batches of matrices or `x.permute(*torch.arange(x.ndim - 1, -1, -1))` to reverse the dimensions of a tensor. (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\aten\src\ATen\native\TensorShape.cpp:4317.)
  X.T.shape


torch.Size([5, 4, 3, 2, 1])

#### Permute

In [42]:
# 1. The Raw Data from the RL Environment
# Shape: [Batch(0), Height(1), Width(2), Frames(3)]
env_states = torch.randn(32, 84, 84, 4) 

# 2. The Permute Fix
# We want: Batch(0), Frames(3), Height(1), Width(2)
pytorch_states = env_states.permute(0, 3, 1, 2)

print(pytorch_states.shape) 

torch.Size([32, 4, 84, 84])


#### Some Best Practices

Changing the order of spatial dimensions requires permute() (or transpose()), never reshape().

Because both transpose and permute change stride metadata without copying data, they leave tensors in a non-contiguous state.
Make it a reflex to chain .contiguous() right after .permute() if you know you need to flatten or reshape it down the line:
- clean_tensor = raw_tensor.permute(0, 3, 1, 2).contiguous() (Creates a new tensor in physical memory if not already contiguous )

## 3. Combining & Splitting

### Concatenate

the only rule is that all dimensions must be equal except the one you intend to concat on

In [44]:
A = torch.zeros(10, 5, 8)

# 1. Glue more sheets on top (dim=0)
B_dim0 = torch.ones(4, 5, 8)
res0 = torch.cat([A, B_dim0], dim=0)
print("dim=0 result:", res0.shape) # [14, 5, 8]

# 2. Glue more rows onto each sheet (dim=1)
B_dim1 = torch.ones(10, 12, 8)
res1 = torch.cat([A, B_dim1], dim=1)
print("dim=1 result:", res1.shape) # [10, 17, 8]

# 3. Glue more columns to each row (dim=2)
B_dim2 = torch.ones(10, 5, 10)
res2 = torch.cat([A, B_dim2], dim=2)
print("dim=2 result:", res2.shape) # [10, 5, 18]

dim=0 result: torch.Size([14, 5, 8])
dim=1 result: torch.Size([10, 17, 8])
dim=2 result: torch.Size([10, 5, 18])


### Stacking

This creates a whole new dimension. For it to work all other dimensions must be equal

In [46]:
X1 = torch.randn(5, 8)
X2 = torch.randn(5, 8)

X_cat=torch.cat([X1, X2], dim=0) # Concatenate along rows (dim=0) 
X_stack=torch.stack([X1, X2], dim=0) # Stack along a new dimension (dim=0)

X_cat.shape, X_stack.shape

(torch.Size([10, 8]), torch.Size([2, 5, 8]))

### Chunking

In [59]:
# Batch of 10 samples with 64 features
x = torch.randn(10, 64)

# 1. Evenly divisible: 10 / 2 = 5 rows each
chunk1, chunk2 = torch.chunk(x, chunks=2, dim=0)
display(chunk1.shape, chunk2.shape)  

# 2. Unevenly divisible: 10 / 3 -> [4, 4, 2]
c1, c2, c3 = torch.chunk(x, chunks=3, dim=0)
c1.shape, c2.shape, c3.shape

torch.Size([5, 64])

torch.Size([5, 64])

(torch.Size([4, 64]), torch.Size([4, 64]), torch.Size([2, 64]))

### Splitting

In [ ]:
x = torch.randn(10, 64)

# "Split into pieces where each piece has at most 3 rows"
splits = torch.split(x, split_size_or_sections=3, dim=0)

# 10 rows split by 3 -> [3, 3, 3, 1]
[s.shape[0] for s in splits]

[3, 3, 3, 1]

In [65]:
x = torch.randn(10, 64)

# "Split into 3 tensors with 2 rows, 5 rows, and 3 rows respectively"
s1, s2, s3 = torch.split(x, split_size_or_sections=[2, 5, 3], dim=0) # must match the total number of rows in x

s1.shape[0], s2.shape[0], s3.shape[0]

(2, 5, 3)

### Some Best Practices

 The difference above between concat and stack is extremely important, if you want to have different samples in one dataset but be as batches, then stacking is the go to. If you want to merge features, or batches if they exist then concatenating is the go to.

Split doesn't throw an error when the split is not divisble, it will throw the largest number it can in the final bin. to avoid unpredictable behavior it is better to use assert before splitting

## 4. Aggregation

In [27]:
X = torch.randn(10000, 10000)

### MIN

In [28]:
X.min()

tensor(-5.4895)

In [29]:
display(X.argmin(dim=0))  # Returns the indices of the minimum values along dimension 0
display(X.argmin(dim=1))  # Returns the indices of the minimum values along dimension 1

tensor([2836,  191, 8440,  ..., 9883, 8004, 9211])

tensor([6064, 8099, 6807,  ..., 3292, 5304, 2970])

### MAX

In [30]:
X.max()

tensor(5.7565)

In [31]:
display(X.argmax(dim=0))  # Returns the indices of the maximum values along dimension 0
display(X.argmax(dim=1))  # Returns the indices of the maximum values along dimension

tensor([7271, 5812, 8649,  ..., 3034, 6511, 2491])

tensor([3217, 6544, 1397,  ..., 8012, 8228, 1550])

### MEAN

In [32]:
display(X.mean())
display(X.mean(dim=0))  # Mean along dimension 0
display(X.mean(dim=1))  # Mean along dimension 1

tensor(-7.9265e-05)

tensor([-0.0078,  0.0087,  0.0091,  ..., -0.0035, -0.0007, -0.0018])

tensor([-0.0116, -0.0035,  0.0054,  ...,  0.0064, -0.0045, -0.0001])

### SUM

In [33]:
display(X.sum())
display(X.sum(dim=0))  # Sum along dimension 0
display(X.sum(dim=1))  # Sum along dimension 1

tensor(-7926.4590)

tensor([-78.1617,  87.2025,  91.0966,  ..., -34.8492,  -7.1375, -17.7447])

tensor([-116.3958,  -35.2897,   54.4238,  ...,   63.6607,  -45.2263,
          -1.0648])

### Some Best Practices

If you want the global argmin or argmax, don't give a dimension but torch will give you the global dimension flattened you have to unravel it to work

In [34]:
X.argmin()

tensor(21290423)

In [35]:
# This will not work
# X[torch.argmin(X)]  # This will raise an error because torch.argmin returns a single index, not a tuple of indices.

X[torch.unravel_index(X.argmin(), X.shape)]

tensor(-5.4895)

 If you want to do the softmax manually, never use log(sum(exp())) since exponentials grow too fast and gpu will throw out a NaN. 
 
 The trick they do is subtract from each number the maximum number in the tensor, and through proof it works perfectly.

Pytorch has that implementation inside torch.logsumexp()

In [36]:
x = torch.tensor([1000.0, 50.0, -10.0])
naive_exp = torch.exp(x)
naive_result = torch.log(torch.sum(naive_exp))
naive_result

tensor(inf)

In [37]:
stable_result = torch.logsumexp(x, dim=0)
stable_result

tensor(1000.)